### Structured output

models can be requested to provide their response in  format matching a given schema. this is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output

### Pydantic

Pydantic model provide richest features set with validation, description and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model
from dotenv import find_dotenv,load_dotenv
load_dotenv(find_dotenv())
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:llama-3.1-8b-instant")
model


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001E4ECF1EBA0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E4ECF1F620>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The Title of the movie")
    year:int=Field(description="the year the movie was released")
    director:str=Field(description="director of the movie")
    rating:float=Field(description="the rating od the movie")

In [3]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001E4ECF1EBA0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E4ECF1F620>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'prop

In [4]:
model.invoke("give me details of movie titanic.")

AIMessage(content='The movie "Titanic" is a 1997 epic romantic disaster film directed, written, co-produced, and co-edited by James Cameron. The film is based on the real-life events of the RMS Titanic\'s sinking in 1912. Here are some key details about the movie:\n\n**Plot**\n\nThe film tells the story of two young lovers, Jack Dawson (Leonardo DiCaprio) and Rose DeWitt Bukater (Kate Winslet), from different social classes, who fall in love aboard the ill-fated ship during its doomed maiden voyage. Jack, a penniless artist, wins his ticket to board the ship in a poker game, while Rose, a high-society woman, is engaged to marry a wealthy man, Cal Hockley (Billy Zane).\n\nAs the ship sets sail, Jack and Rose form a deep connection, and Rose begins to question her engagement and her place in society. Meanwhile, the ship\'s crew and passengers are unaware of the danger that lies ahead, as the Titanic is approaching an iceberg in the North Atlantic.\n\n**Cast**\n\n* Leonardo DiCaprio as Ja

In [5]:
model_with_structure.invoke("give me details of movie 2012.")

Movie(title='2012', year=2009, director='Roland Emmerich', rating=6.4)

### Message output along parsed structure

In [6]:
from pydantic import  BaseModel,Field

class Movie(BaseModel):
    title: str = Field(...,description="this is the title of the movie?")
    year: int = Field(..., description="year the movie was released")
    director:str=Field(...,description="director of the movie")
    rating:float=Field(...,description="the rating od the movie")
    genre: str = Field(...,description="the type of movie")

mdoel_with_structure_and_raw = model.with_structured_output(Movie, include_raw=True)
response = mdoel_with_structure_and_raw.invoke("give me details about movie rio?")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'fejp3880t', 'function': {'arguments': '{"director":"Carlos Saldanha","genre":"family","rating":7.5,"title":"Rio","year":2011}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 293, 'total_tokens': 334, 'completion_time': 0.046019706, 'completion_tokens_details': None, 'prompt_time': 0.016490392, 'prompt_tokens_details': None, 'queue_time': 0.158882886, 'total_time': 0.062510098}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f6a6b-ae2b-7071-9004-19fd0221a932-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Carlos Saldanha', 'genre': 'family', 'rating': 7.5, 'title': 'Rio', 'year': 2011}, 'id': 'fejp3880t', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 293, 'ou

### Nested structure

In [17]:
from pydantic import  BaseModel, Field

class Actor(BaseModel):
    name:str 
    role:str 

class MovieDetails(BaseModel):
     title: str
     year: int
     director:str
     rating:float
     genres: list[str]
     cast:list[Actor]
     budget:str

In [18]:
model_with_structure_details= model.with_structured_output(MovieDetails)
response = model_with_structure_details.invoke("tell me about the movies spiderman")
response

MovieDetails(title='Spiderman', year=2002, director='Sam Raimi', rating=7.9, genres=['Action', 'Adventure', 'Sci-Fi'], cast=[Actor(name='Tobey Maguire', role='Spiderman/Peter Parker'), Actor(name='Willem Dafoe', role='Norman Osborn/Green Goblin'), Actor(name='Kristen Dunst', role='Mary Jane Watson')], budget='139 million')

### TypedDict

typeDict provides a simple alternative using python's  built-in typing  , ideal when you do not need runtime validation

In [20]:
from typing_extensions import TypedDict,Annotated

class movieDict(TypedDict):
    name: Annotated[str, ... ," the name of the movie"]
    year:Annotated[int, ...,"the year it was released"]
    director:Annotated[str, ...,"the director of the movie"]
    rating:Annotated[float, ...,"rating of the movie out of 10"]

model_with_typedict = model.with_structured_output(movieDict)
response = model_with_typedict.invoke("tell me abput the movie ironman")
response

{'director': 'Jon Favreau', 'name': 'Iron Man', 'rating': 7.9, 'year': 2008}

In [30]:
class Actor(TypedDict):
    name:str 
    role:str 

class MovieDetails(TypedDict):
     title: str
     year: int
     director:str
     rating:float
     genres: list[str]
     cast:list[Actor]
     budget:str

model_with_structure_detail= model.with_structured_output(MovieDetails)
response = model_with_structure_detail.invoke("tell me about the movies spiderman")
response

{'budget': '280000000',
 'cast': [{'name': 'Tobey Maguire', 'role': 'Peter Parker/Spider-Man'},
  {'name': 'Willem Dafoe', 'role': 'Norman Osborn/Green Goblin'}],
 'director': 'Sam Raimi',
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'rating': 7.7,
 'title': 'Spider-Man',
 'year': 2002}

In [31]:
model.profile

{'name': 'Llama 3.1 8B Instant',
 'release_date': '2024-07-23',
 'last_updated': '2024-07-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 131072,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

### DATA CLASS

a data class is a class typically containing minly data, although there are not really any restriction , you create it using the @dataclass decorator

In [ ]:
# using PYDANTIC
from pydantic import BaseModel, Field 
from langchain.agents import create_agent

class contactInfo(BaseModel):
    name: str = Field(description="The name of the person")
    email:  str = Field(description="the email of the person")
    contact:  str = Field(description="contact of the person")


agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    response_format=contactInfo
)

response = agent.invoke(
    {"messages":[{"role": "user","content":"Extract the contact info from the: jhon doe , jhon@example.com , +1 (755) 123-4567 "}]}
)

print(response["structured_response"])

name='jhon doe' email='jhon@example.com' contact='+1 (755) 123-4567'


In [ ]:
# Using TypedDict
from typing import TypedDict, Annotated
from langchain.agents import create_agent

class contactinfo(TypedDict):
    name: Annotated[str, ... , "the name of the person"] 
    email:Annotated[str, ... , "the email of the person"]
    contact: Annotated[str, ... , "the contact number of the person"] 


agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    response_format=contactinfo
)

response = agent.invoke(
    {"messages":[{"role": "user","content":"Extract the contact info from the: jhon doe , jhon@example.com , +1 (755) 123-4567 "}]}
)

print(response["structured_response"])

{'name': 'jhon doe', 'email': 'jhon@example.com', 'contact': '+1 (755) 123-4567'}


In [48]:
## using DATACLASS

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContInfo:
    """contact info of a person """
    name: str # name of the person
    eamil: str # email of the person
    caontact: str # contact of the person

agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    response_format=ContInfo
) 

response = agent.invoke(
    {"messages":[{"role":"user","content":"Extract the contact info from the: jhon doe , jhon@example.com , +1 (755) 123-4567" }]}
)

print(response["structured_response"])

ContInfo(name='jhon doe', eamil='jhon@example.com', caontact='+1 (755) 123-4567')
